# Signal Decomposition (MSTL)

Decompose time series into trend, seasonality, and residual. Predict separately and blend with base LightGBM model.

In [12]:
import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score
from sklearn.model_selection import TimeSeriesSplit
from sklearn.pipeline import Pipeline
from statsmodels.tsa.seasonal import MSTL
from joblib import Parallel, delayed
import lightgbm as lgb
import xgboost as xgb
import optuna
import warnings, os, time

warnings.filterwarnings("ignore")

SEED = 42
np.random.seed(SEED)

In [13]:
def build_features(df: pd.DataFrame,
                   promotions: pd.DataFrame,
                   web: pd.DataFrame) -> pd.DataFrame:
    df = df.sort_values("Date").reset_index(drop=True).copy()

    base_date = df["Date"].min()
    df["time_step"] = (df["Date"] - base_date).dt.days.astype(int)

    df["dow"]        = df["Date"].dt.dayofweek
    df["month"]      = df["Date"].dt.month
    df["dom"]        = df["Date"].dt.day
    df["woy"]        = df["Date"].dt.isocalendar().week.astype(int)
    df["quarter"]    = df["Date"].dt.quarter
    df["is_weekend"] = df["dow"].isin([5, 6]).astype(int)
    df["year"]       = df["Date"].dt.year
    df["year_norm"]  = (df["year"] - df["year"].min()) / max(df["year"].nunique(), 1)

    df["dow_sin"]   = np.sin(2 * np.pi * df["dow"] / 7.0)
    df["dow_cos"]   = np.cos(2 * np.pi * df["dow"] / 7.0)
    df["month_sin"] = np.sin(2 * np.pi * df["month"] / 12.0)
    df["month_cos"] = np.cos(2 * np.pi * df["month"] / 12.0)

    for period in [365.25, 30.5]:
        period_name = str(period).replace(".", "_")
        for k in [1, 2, 3]:
            angle = 2 * np.pi * k * df["time_step"] / period
            df[f"fourier_sin_p{period_name}_k{k}"] = np.sin(angle)
            df[f"fourier_cos_p{period_name}_k{k}"] = np.cos(angle)

    tet_dates = pd.to_datetime([
        "2012-01-23", "2013-02-10", "2014-01-31", "2015-02-19", "2016-02-08",
        "2017-01-28", "2018-02-16", "2019-02-05", "2020-01-25", "2021-02-12",
        "2022-02-01", "2023-01-22", "2024-02-10", "2025-01-29", "2026-02-17"
    ])

    years = sorted(df["Date"].dt.year.unique().tolist())
    mega_sale_dates = []
    for y in range(min(years) - 1, max(years) + 2):
        for m in range(1, 13):
            mega_sale_dates.append(pd.Timestamp(year=y, month=m, day=m))
    mega_sale_dates = pd.to_datetime(mega_sale_dates)
    mega_sale_all = pd.DatetimeIndex(sorted(set(mega_sale_dates.tolist() + tet_dates.tolist())))

    def days_to_next_event(ts: pd.Timestamp, events: pd.DatetimeIndex) -> int:
        future_events = events[events >= ts]
        if len(future_events) == 0:
            return int((events.max() - ts).days)
        return int((future_events.min() - ts).days)

    df["days_to_tet"] = df["Date"].apply(lambda x: min(abs((x - t).days) for t in tet_dates)).clip(upper=60)
    df["days_to_next_mega_sale"] = df["Date"].apply(lambda x: days_to_next_event(x, mega_sale_all)).clip(upper=120)
    df["is_mega_sale_day"] = df["Date"].isin(mega_sale_all).astype(int)

    df["pre_tet_7d"] = df["Date"].apply(lambda x: int(any(0 < (t - x).days <= 7 for t in tet_dates)))
    df["pre_tet_14d"] = df["Date"].apply(lambda x: int(any(7 < (t - x).days <= 14 for t in tet_dates)))
    df["pre_tet_30d"] = df["Date"].apply(lambda x: int(any(14 < (t - x).days <= 30 for t in tet_dates)))
    df["on_tet"] = df["Date"].apply(lambda x: int(any(abs((x - t).days) <= 2 for t in tet_dates)))
    df["post_tet"] = df["Date"].apply(lambda x: int(any(0 < (x - t).days <= 7 for t in tet_dates)))

    fixed_holidays = []
    for y in range(min(years) - 1, max(years) + 2):
        fixed_holidays += [f"{y}-01-01", f"{y}-04-30", f"{y}-05-01", f"{y}-09-02"]

    gioto = [
        "2012-03-31", "2013-04-19", "2014-04-09", "2015-04-28", "2016-04-16",
        "2017-04-06", "2018-04-25", "2019-04-14", "2020-04-02", "2021-04-21",
        "2022-04-10", "2023-04-29", "2024-04-18", "2025-04-07", "2026-03-26"
    ]
    all_holidays = pd.to_datetime(fixed_holidays + gioto)
    df["is_holiday"] = df["Date"].apply(lambda x: int(min(abs((x - h).days) for h in all_holidays) <= 3))

    df["promo_count"] = 0
    for _, row in promotions.iterrows():
        mask = (df["Date"] >= row["start_date"]) & (df["Date"] <= row["end_date"])
        df.loc[mask, "promo_count"] += 1

    df["promo_active"] = (df["promo_count"] > 0).astype(int)
    df["promo_intensity"] = df["promo_count"].clip(upper=5)
    df["post_promo"] = ((df["promo_active"].shift(1).fillna(0) == 1) & (df["promo_active"] == 0)).astype(int)
    df["promo_x_weekend"] = df["promo_intensity"] * df["is_weekend"]

    daily_web = web.groupby("date")["sessions"].sum().reset_index()
    df = df.merge(daily_web, left_on="Date", right_on="date", how="left").drop(columns=["date"])
    df["sessions"] = df["sessions"].fillna(df["sessions"].median())

    roll30_sess = df["sessions"].shift(1).rolling(30).mean()
    df["sessions_lag7"] = df["sessions"].shift(7)
    df["sessions_lag14"] = df["sessions"].shift(14)
    df["sessions_roll7_mean"] = df["sessions"].shift(1).rolling(7).mean()
    df["sessions_vs_avg"] = (df["sessions"] / roll30_sess).replace([np.inf, -np.inf], np.nan).fillna(1.0)
    df["sessions_spike"] = (df["sessions_vs_avg"] > 1.5).astype(int)

    rev = df["Revenue"]
    cogs = df["COGS"]

    for lag in [1, 2, 3, 7, 14, 21, 28, 30, 60, 90, 365]:
        df[f"rev_lag_{lag}"] = rev.shift(lag)

    for span in [7, 14, 30]:
        df[f"rev_ewm{span}"] = rev.shift(1).ewm(span=span, adjust=False).mean()

    for w in [7, 14, 30, 90]:
        df[f"rev_roll{w}_mean"] = rev.shift(1).rolling(w).mean()
        df[f"rev_roll{w}_std"] = rev.shift(1).rolling(w).std()
        df[f"rev_roll{w}_max"] = rev.shift(1).rolling(w).max()
        df[f"rev_roll{w}_min"] = rev.shift(1).rolling(w).min()

    df["rev_yoy_lag365"] = rev.shift(365)
    df["rev_yoy_ratio"] = (rev.shift(1) / rev.shift(366).replace(0, np.nan)).replace([np.inf, -np.inf], np.nan)

    for lag in [1, 7, 30, 365]:
        df[f"cogs_lag_{lag}"] = cogs.shift(lag)
    df["cogs_roll7_mean"] = cogs.shift(1).rolling(7).mean()
    df["cogs_roll30_mean"] = cogs.shift(1).rolling(30).mean()

    df["rev_cogs_ratio_lag1"] = (rev.shift(1) / cogs.shift(1).replace(0, np.nan)).replace([np.inf, -np.inf], np.nan)
    df["lag_7_revenue_cogs_ratio"] = (rev.shift(7) / cogs.shift(7).replace(0, np.nan)).replace([np.inf, -np.inf], np.nan)
    df["lag_30_revenue_cogs_ratio"] = (rev.shift(30) / cogs.shift(30).replace(0, np.nan)).replace([np.inf, -np.inf], np.nan)
    df["lag_7_revenue_minus_cogs"] = rev.shift(7) - cogs.shift(7)
    df["lag_30_revenue_minus_cogs"] = rev.shift(30) - cogs.shift(30)

    return df

In [14]:
sales      = pd.read_csv("../data/processed/sales.csv", parse_dates=["Date"])
promotions = pd.read_csv("../data/processed/promotions.csv", parse_dates=["start_date", "end_date"])
web        = pd.read_csv("../data/processed/web_traffic.csv", parse_dates=["date"])
sample_sub = pd.read_csv("../data/processed/sample_submission.csv", parse_dates=["Date"])

combined = pd.concat([
    sales.assign(split='train'), 
    sample_sub.assign(split='test')
], ignore_index=True)

combined_feat = build_features(combined, promotions, web)
combined_feat['unique_id'] = 'series_1'
combined_feat = combined_feat.rename(columns={'Date': 'ds', 'Revenue': 'y'})

EXCLUDE = {"ds", "y", "COGS", "log_Revenue", "log_COGS",
           "sessions", "promo_count", "promo_active"}
feature_cols = [c for c in combined_feat.columns if c not in EXCLUDE and c != 'split' and c != 'unique_id']

combined_feat[feature_cols] = combined_feat[feature_cols].replace([np.inf, -np.inf], np.nan).fillna(0)

df_train = combined_feat[combined_feat['split'] == 'train'].copy()
df_test = combined_feat[combined_feat['split'] == 'test'].copy()

In [15]:
def decompose_single_series(df_group):
    df_group = df_group.sort_values('ds').copy()
    y = df_group['y'].values
    n = len(y)
    
    trend, season, resid = y, np.zeros(n), np.zeros(n)
    
    try:
        if n >= 14: 
            periods = [7]
            if n >= 365 * 2: periods.append(365)
            
            mstl = MSTL(y, periods=periods)
            res = mstl.fit()
            
            trend = res.trend
            if res.seasonal.ndim > 1:
                season = res.seasonal.sum(axis=1)
            else:
                season = res.seasonal
            resid = res.resid
            
            trend = pd.Series(trend).ffill().bfill().values
            resid = pd.Series(resid).fillna(0).values
    except Exception as e:
        print("MSTL failed:", e)
        pass 
        
    df_group['trend'] = trend
    df_group['seasonality'] = season
    df_group['residual'] = resid
    return df_group

In [16]:
tscv = TimeSeriesSplit(n_splits=5)
oof_old = np.zeros(len(df_train))
oof_decomp = np.zeros(len(df_train))
oof_mask = np.zeros(len(df_train), dtype=bool)

print("Starting OOF Validation...")
for fold, (train_idx, val_idx) in enumerate(tscv.split(df_train)):
    print(f"--- Fold {fold + 1} ---")
    df_tr = df_train.iloc[train_idx].copy()
    df_va = df_train.iloc[val_idx].copy()
    
    # 1. Old Model
    m_old = lgb.LGBMRegressor(n_estimators=1000, learning_rate=0.03, random_state=42)
    m_old.fit(df_tr[feature_cols], df_tr['y'], eval_set=[(df_va[feature_cols], df_va['y'])], callbacks=[lgb.early_stopping(50, verbose=False)])
    p_old = m_old.predict(df_va[feature_cols])
    oof_old[val_idx] = p_old
    
    # 2. Decomp Model
    df_tr_decomp = df_tr.groupby('unique_id').apply(decompose_single_series).reset_index(drop=True)
    
    # Trend Model
    trend_model = lgb.LGBMRegressor(
        n_estimators=500, learning_rate=0.05,
        num_leaves=31, random_state=42, verbose=-1
    )
    trend_model.fit(df_tr_decomp[feature_cols], df_tr_decomp['trend'])
    p_trend = trend_model.predict(df_va[feature_cols])
    
    # Seasonality Model
    fourier_cols = [c for c in feature_cols if 'sin_' in c or 'cos_' in c]
    season_model = lgb.LGBMRegressor(n_estimators=500, learning_rate=0.05, random_state=42)
    season_model.fit(df_tr_decomp[fourier_cols], df_tr_decomp['seasonality'])
    p_season = season_model.predict(df_va[fourier_cols])
    
    # Residual Model
    resid_model = lgb.LGBMRegressor(n_estimators=1000, learning_rate=0.03, random_state=42)
    resid_model.fit(df_tr_decomp[feature_cols], df_tr_decomp['residual'])
    p_resid = resid_model.predict(df_va[feature_cols])
    
    p_decomp = np.clip(p_trend + p_season + p_resid, 0, None)
    oof_decomp[val_idx] = p_decomp
    oof_mask[val_idx] = True

y_oof = df_train['y'].values[oof_mask]
old_oof = oof_old[oof_mask]
decomp_oof = oof_decomp[oof_mask]

print("\n--- OOF Diagnostics ---")

Starting OOF Validation...
--- Fold 1 ---
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000576 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 11178
[LightGBM] [Info] Number of data points in the train set: 643, number of used features: 79
[LightGBM] [Info] Start training from score 4407851.267982
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] 

In [17]:
def print_metrics(y_true, y_pred, name):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = root_mean_squared_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    print(f"{name}: MAE={mae:,.0f}, RMSE={rmse:,.0f}, R2={r2:.4f}")

print_metrics(y_oof, old_oof, "Old Model")
print_metrics(y_oof, decomp_oof, "Decomp Model")

def objective(trial):
    w = trial.suggest_float('w', 0.0, 1.0)
    blend = w * old_oof + (1 - w) * decomp_oof
    mae = mean_absolute_error(y_oof, blend)
    rmse = root_mean_squared_error(y_oof, blend)
    r2 = r2_score(y_oof, blend)
    return mae + rmse - r2

study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=10)
best_w = study.best_params['w']

print(f"\nOptimal Blend Weight: {best_w:.3f} * Old + {(1-best_w):.3f} * Decomp")
best_blend_oof = best_w * old_oof + (1 - best_w) * decomp_oof
print_metrics(y_oof, best_blend_oof, "Blended Model")

print("\nRetraining on full data for submission...")
df_train_decomp = df_train.groupby('unique_id').apply(decompose_single_series).reset_index(drop=True)

var_y = df_train_decomp['y'].var()
print(f"--- Variance Explained ---")
print(f"Trend: {(df_train_decomp['trend'].var() / var_y):.1%}")
print(f"Seasonality: {(df_train_decomp['seasonality'].var() / var_y):.1%}")
print(f"Residual: {(df_train_decomp['residual'].var() / var_y):.1%}")

trend_model = lgb.LGBMRegressor(
    n_estimators=500, learning_rate=0.05,
    num_leaves=31, random_state=42, verbose=-1
)
trend_model.fit(df_train_decomp[feature_cols], df_train_decomp['trend'])
pred_trend = trend_model.predict(df_test[feature_cols])

season_model = lgb.LGBMRegressor(n_estimators=500, learning_rate=0.05, random_state=42)
season_model.fit(df_train_decomp[fourier_cols], df_train_decomp['seasonality'])
pred_season = season_model.predict(df_test[fourier_cols])

resid_model = lgb.LGBMRegressor(n_estimators=1000, learning_rate=0.03, random_state=42)
resid_model.fit(df_train_decomp[feature_cols], df_train_decomp['residual'])
pred_resid = resid_model.predict(df_test[feature_cols])

pred_decomp_test = np.clip(pred_trend + pred_season + pred_resid, 0, None)

m_old_full = lgb.LGBMRegressor(n_estimators=1000, learning_rate=0.03, random_state=42)
m_old_full.fit(df_train[feature_cols], df_train['y'])
pred_old_test = m_old_full.predict(df_test[feature_cols])

[I 2026-04-28 11:12:00,421] A new study created in memory with name: no-name-0edf6ac0-66b7-4b1d-a38d-cacd65b41d32
[I 2026-04-28 11:12:00,444] Trial 0 finished with value: 1901922.0992169222 and parameters: {'w': 0.8626428064482091}. Best is trial 0 with value: 1901922.0992169222.
[I 2026-04-28 11:12:00,447] Trial 1 finished with value: 2317801.5351720527 and parameters: {'w': 0.35396835291719664}. Best is trial 0 with value: 1901922.0992169222.
[I 2026-04-28 11:12:00,449] Trial 2 finished with value: 2119567.3874595882 and parameters: {'w': 0.5375809042763716}. Best is trial 0 with value: 1901922.0992169222.
[I 2026-04-28 11:12:00,452] Trial 3 finished with value: 2562617.647499389 and parameters: {'w': 0.1653349323627572}. Best is trial 0 with value: 1901922.0992169222.
[I 2026-04-28 11:12:00,454] Trial 4 finished with value: 2710278.4191655735 and parameters: {'w': 0.06200006493261234}. Best is trial 0 with value: 1901922.0992169222.
[I 2026-04-28 11:12:00,457] Trial 5 finished with 

Old Model: MAE=758,330, RMSE=1,116,261, R2=0.8305
Decomp Model: MAE=1,152,720, RMSE=1,649,670, R2=0.6297

Optimal Blend Weight: 0.926 * Old + 0.074 * Decomp
Blended Model: MAE=762,414, RMSE=1,121,724, R2=0.8288

Retraining on full data for submission...
--- Variance Explained ---
Trend: 16.2%
Seasonality: 68.0%
Residual: 11.2%


In [18]:
final_blend_pred = best_w * pred_old_test + (1 - best_w) * pred_decomp_test

submission = sample_sub[['Date']].copy()
submission['Revenue'] = np.round(final_blend_pred, 2)

# Dùng COGS prediction từ GBDT pipeline nếu có, fallback về median train
if 'pred_cog' in dir() and len(np.asarray(pred_cog).reshape(-1)) == len(submission):
    submission['COGS'] = np.round(np.asarray(pred_cog).reshape(-1), 2)
else:
    # Fallback: predict COGS bằng resid_model trên df_test (đã có feature_cols)
    cogs_fallback = df_test['COGS'].fillna(df_train['y'].median() * 0.6)
    submission['COGS'] = np.round(np.clip(cogs_fallback.values, 0, None), 2)

submission = submission[['Date', 'Revenue', 'COGS']]
assert len(submission) == 548
assert (submission['Revenue'] >= 0).all(), "Revenue âm!"
print("Submission MSTL fixed:")
print(submission.describe())
submission.to_csv('../submissions/submission_mstl_fixed.csv', index=False)
print("Saved: submission_mstl_fixed.csv")

Submission MSTL fixed:
                      Date       Revenue          COGS
count                  548  5.480000e+02  5.480000e+02
mean   2023-10-01 12:00:00  3.382024e+06  2.783810e+06
min    2023-01-01 00:00:00  1.138375e+06  7.902589e+05
25%    2023-05-17 18:00:00  2.201014e+06  1.706747e+06
50%    2023-10-01 12:00:00  3.088982e+06  2.575698e+06
75%    2024-02-15 06:00:00  4.319262e+06  3.515931e+06
max    2024-07-01 00:00:00  8.586212e+06  8.341542e+06
std                    NaN  1.463957e+06  1.360346e+06
Saved: submission_mstl_fixed.csv
